# Pipeline For The Structure Preserving Method (BDI)
$H \to DLA \to e^{i \rho t} \to BDI \to U_{\text{reconstructed}}$

## Imports

In [ ]:
import sys, os
# Make the repo root importable so the `functions` package is found
_root = os.getcwd()
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, "functions")):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

import numpy as np


## Choose model and decomposition parameters
This cell sets the TFIM model size and evolution time.
You can change n, J, h, t and periodic according to your experiment. We remind that we are working with the Hamiltonian:
$
\begin{equation}
    H = -J \sum^{N-2}_{i=0}Z_iZ_{i+1} -h \sum^{N-1}_{i=0}X_i,
\end{equation}
$

In [ ]:
# Model parameters
n = 2
J = 1.0
h = 1.0
t = 0.5
rotated = True
periodic = False

print(f"n = {n}, J = {J}, h = {h}, t = {t}, rotated = {rotated}, periodic = {periodic}")

n = 2, J = 1.0, h = 1.0, t = 0.5, rotated = True, periodic = False


In [ ]:
from functions.common.build_TFIM import TFIM_Ham

if n <= 10:
    hamiltonian = TFIM_Ham(n)
    print(f"Hamiltonian shape: {hamiltonian.shape}")
    print(hamiltonian)

Hamiltonian shape: (4, 4)
[[-2.  0.  0. -1.]
 [ 0.  0. -1.  0.]
 [ 0. -1.  0.  0.]
 [-1.  0.  0.  2.]]


## Build DLA generators (Pauli-word form)
We generate TFIM Pauli words and grow the DLA closure.

In [ ]:
from functions.structure_preserving.find_DLA import tfim_pauliwords_gen, dla_pauli_words

generators = tfim_pauliwords_gen(n, rotated=rotated, periodic=periodic)
dla_words = dla_pauli_words(generators)

print("Initial generators:", len(generators))
print("DLA size:", len(dla_words))
print("First few DLA words:", dla_words)

Initial generators: 3
DLA size: 6
First few DLA words: [('X', 'X'), ('Z', 'I'), ('I', 'Z'), ('Y', 'X'), ('X', 'Y'), ('Y', 'Y')]


## Map to the Majorana/isomorphic so(2n) matrix
This creates the isomorphism $\rho(iH)$ used before exponentiation.

In [ ]:
from functions.structure_preserving.build_isomorphism import map_to_majarana, build_so_matrix

maj_mapping = map_to_majarana(generators, J=J, h=h)
print(maj_mapping)
rho_iH = build_so_matrix(maj_mapping, n)

print("rho(iH) shape:", rho_iH.shape)
print("rho(iH) skew Hermitian?", np.allclose(rho_iH.T, -rho_iH, atol=1e-10))
print(rho_iH)

{('X', 'X'): (1.0, (0, 3)), ('Z', 'I'): (-1.0, (0, 1)), ('I', 'Z'): (-1.0, (2, 3))}
rho(iH) shape: (4, 4)
rho(iH) skew Hermitian? True
[[ 0. -2.  0.  2.]
 [ 2.  0.  0.  0.]
 [ 0.  0.  0. -2.]
 [-2.  0.  2.  0.]]


## Exponentiate to get the matrix to decompose
Now build $U(t)=\exp(t\,\rho(iH))$ and verify unitarity/orthogonality.

In [ ]:
from functions.structure_preserving.BDI_decomp import from_generator

U_t = from_generator(rho_iH, t=t)

print("U(t) shape:", U_t.shape)
print("Is U(t) unitary?", np.allclose(U_t @ U_t.conj().T, np.eye(U_t.shape[0]), atol=1e-10))
print("Is U(t) real (orthogonal case)?", np.allclose(U_t.imag, 0.0, atol=1e-10))

U(t) shape: (4, 4)
Is U(t) unitary? True
Is U(t) real (orthogonal case)? True


## One-step BDI KAK decomposition
Decompose $U(t)$ one step, rebuild $K_1 A K_2$, and check reconstruction error to verify BDI.

In [ ]:
from functions.structure_preserving.BDI_decomp import bdi, build_kak

k11, k12, theta, k21, k22 = bdi(U_t)
K1, A, K2 = build_kak(k11, k12, theta, k21, k22)
U_rec = K1 @ A @ K2

print("K1 shape:", K1.shape, "A shape:", A.shape, "K2 shape:", K2.shape)
print("One-step reconstruction allclose:", np.allclose(U_t, U_rec, atol=1e-10))
print("Max reconstruction error:", np.max(np.abs(U_t - U_rec)))

K1 shape: (4, 4) A shape: (4, 4) K2 shape: (4, 4)
One-step reconstruction allclose: True
Max reconstruction error: 2.7755575615628914e-16


## Recursive BDI
Run the recursive factorization and print operation counts by type.

In [ ]:
from functions.structure_preserving.BDI_decomp import recursive_bdi

ops = recursive_bdi(U_t, num_iter=None, return_all=False) # This cell needs return_all=False to get the final list of ops
print("Total recursive ops:", len(ops))

types = {}
for _, _, _, op_type in ops:
    types[op_type] = types.get(op_type, 0) + 1

print("Operation type counts:")
for k in sorted(types, key=lambda x: str(x)):
    print(f" {k}: {types[k]}")

print(ops)


Total recursive ops: 5
Operation type counts:
 a0: 1
 k1: 2
 k2: 2
[(array([[ 0.5109844 ,  0.85958999],
       [-0.85958999,  0.5109844 ]]), 0, 2, 'k1'), (array([[-0.85958999,  0.5109844 ],
       [-0.5109844 , -0.85958999]]), 2, 4, 'k1'), (array([0.08613249, 0.91386751]), 0, 4, 'a0'), (array([[-0.5109844 , -0.85958999],
       [ 0.85958999, -0.5109844 ]]), 0, 2, 'k2'), (array([[-0.85958999,  0.5109844 ],
       [-0.5109844 , -0.85958999]]), 2, 4, 'k2')]


In [ ]:
from functions.structure_preserving.BDI_verification import verify_bdi_decomposition
verify_bdi_decomposition(U_t, ops)

Max reconstruction error: 2.7755575615628914e-16
Recursive allclose: True


In [ ]:
# Print out the determinants of the K1, A, K2 to check if they are in SO(2n) or O(2n)
det_1 = 0
det_m1 = 0
for element in ops:
    if element[3] in ['k1', 'k2']:
        det = np.linalg.det(element[0])
        #print(f"{element[3]} determinant: {det:.4f}")
        if det > 0.5:
            det_1 += 1
        else:
            det_m1 += 1
print(f"Number of K1/K2 with determinant +1: {det_1}")
print(f"Number of K1/K2 with determinant -1: {det_m1}")

Number of K1/K2 with determinant +1: 4
Number of K1/K2 with determinant -1: 0


## Map Back to Pauli Rotations

In [ ]:
from functions.structure_preserving.map_back import pw_to_majorana, build_majorana_dla_map, map_ops_to_pauli


In [ ]:
mapping = build_majorana_dla_map(dla_words)
print(f"Majorana DLA map size: {len(mapping)}")
print(mapping)


Majorana DLA map size: 6


In [ ]:
from collections import Counter

pauli_decomp = map_ops_to_pauli(ops, mapping, time=t)

print(f"Number of Pauli rotations: {len(pauli_decomp)}")
print(f"Expected number:           {n*(2*n-1)}")
print()

print("Pauli rotations:")
print("-" * 45)
print(f"{'#':>2}  {'stage':>5}  {'Pauli':>8}  {'alpha':>12}")
print("-" * 45)

for i, (word, coeff, op_type) in enumerate(pauli_decomp):
    word_str ="".join(word)
    print(f"{i:>2}  {str(op_type):>5}  {word_str:>8}  {coeff:>12.6f}")

Number of Pauli rotations: 6
Expected number:           6

Pauli rotations:
---------------------------------------------
 #  stage     Pauli         alpha
---------------------------------------------
 0     k1        ZI      0.517233
 1     k1        IZ      1.302632
 2     a0        XY      0.086132
 3     a0        YX     -0.913868
 4     k2        ZI     -1.053563
 5     k2        IZ      1.302632


## Verify Reconstruction

In [ ]:
from scipy.linalg import expm
from functions.common.verification import (
    pauli_word_to_matrix,
    phase_aligned_error,
    reconstruct_pauli_decomp,
)


In [ ]:
# Reconstruct U(t) from the Pauli rotations and compare to exp(-i H t)
if n <= 7:
    U_ref = expm(-1j * hamiltonian * t)
    U_rec = reconstruct_pauli_decomp(pauli_decomp, n, t)
    err = phase_aligned_error(U_ref, U_rec)
    print(f"Number of Pauli rotations: {len(pauli_decomp)} (expected {n * (2*n - 1)})")
    print(f"Phase-aligned reconstruction error: {err:.3e}")
    print("PASS" if err < 1e-8 else "FAIL")
else:
    print(f"n={n}: full Hilbert-space verification skipped.")


In [ ]:
import pennylane as qml
import matplotlib.pyplot as plt


def kak_circuit(pauli_decomp, time):
    """Apply the Pauli decomposition as PennyLane PauliRot gates.

    PauliRot(phi, P) = exp(-i phi P / 2), so phi = 2*alpha for exp(-i alpha P).
    The 'a0' coefficients are stored time-normalised, so they are multiplied by
    time here. Gates are applied in reverse so the circuit matrix matches
    U = G_0 @ G_1 @ ... @ G_K.
    """
    for word, coeff, op_type in pauli_decomp[::-1]:
        alpha = coeff * time if op_type == "a0" else coeff
        pauli_str = "".join(p for p in word if p != "I")
        wires = [i for i, p in enumerate(word) if p != "I"]
        qml.PauliRot(2 * alpha, pauli_str, wires=wires)


dev = qml.device("default.qubit", wires=n)


@qml.qnode(dev)
def circuit(time):
    kak_circuit(pauli_decomp, time)
    return qml.state()


In [ ]:
# Verify the PennyLane circuit and draw it
if n <= 7:
    U_ref = expm(-1j * hamiltonian * t)
    U_circ = qml.matrix(circuit)(t)
    err = phase_aligned_error(U_ref, U_circ)
    print(f"PennyLane circuit phase-aligned error: {err:.3e}")
    print("PASS" if err < 1e-8 else "FAIL")
    qml.draw_mpl(circuit)(t)
    plt.show()
else:
    print(f"n={n}: circuit verification skipped.")
